# 医学多智能体 + RAG MVP（Colab / Cursor 远程内核）

推荐策略：**代码走 GitHub，数据走 Google Drive**（大图片、FAISS、JSON 不进 Git）。

1. 先在本机把**整个项目仓库**（含 `medical_mvp/` 与 `requirements.txt`）推到 GitHub；下面第一格会 `clone`/`pull` 到 `/content/...`。
2. 在 Drive 中建好数据目录（如 `MyDrive/毕业设计/medical_data`），第一格会把 `MEDICAL_MVP_DATA_ROOT` 指到该路径，与 `config.py` 一致。
3. 设置 `GOOGLE_API_KEY`（环境变量或 Colab「密钥」）后依次运行后续单元。

In [ ]:
%pip install -q google-genai datasets sentence-transformers faiss-cpu Pillow tqdm

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- 按需修改以下常量 ---
GITHUB_REPO_URL = "https://github.com/你的用户名/你的仓库名.git"  # 整个项目根（内含 medical_mvp 包）
CLONE_PARENT = Path("/content")
REPO_FOLDER_NAME = "project"  # clone 后生成的目录名，需与 GitHub 仓库名一致（或 clone 后看 /content 下实际文件夹名）
DATA_ROOT = Path("/content/drive/MyDrive/毕业设计/medical_data")  # Drive 上持久化数据目录

# 1) 挂载 Drive（仅在 Colab / Colab 远程内核中可用）
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["MEDICAL_MVP_DATA_ROOT"] = str(DATA_ROOT)

# 2) 拉取代码：首次 clone，之后 pull（避免 ! 与 if 缩进混用，统一用 subprocess）
CODE_ROOT = CLONE_PARENT / REPO_FOLDER_NAME
if not CODE_ROOT.is_dir():
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(CODE_ROOT)], check=True, cwd=str(CLONE_PARENT))
else:
    subprocess.run(["git", "pull"], check=True, cwd=str(CODE_ROOT))

sys.path.insert(0, str(CODE_ROOT))
os.chdir(CODE_ROOT)

print("CODE_ROOT =", CODE_ROOT)
print("MEDICAL_MVP_DATA_ROOT =", os.environ["MEDICAL_MVP_DATA_ROOT"])

In [ ]:
import os

# 在 Colab 网页版可用「密钥」注入；本地调试请自行 export / 或在下方临时赋值（勿提交到 Git）
# os.environ["GOOGLE_API_KEY"] = "..."
if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("缺少 GOOGLE_API_KEY：请在运行 Vision 相关单元前设置该环境变量。")

In [ ]:
# Drive 已在首格挂载，数据目录已由 MEDICAL_MVP_DATA_ROOT 指定，无需再 mount
from medical_mvp.data_preparation import stream_pmc_vqa_and_build_database

stream_pmc_vqa_and_build_database(limit=200)

In [ ]:
from medical_mvp.run_mvp import run_random_samples

run_random_samples(n=3, seed=42)